Setup and Import

In [ ]:
import numpy as np
import tensorflow as tf
import os
import json
import time
import nltk
import pandas as pd
import sys
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score

# Download dependensi METEOR
nltk.download('wordnet', quiet=True)
nltk.download('punkt', quiet=True)

sys.path.append(os.path.abspath("../../")) 
from rnn_lstm.decoder_rnn import DecoderRNN

# Load metadata
with open('../../../dataset/vocab.json', 'r') as f:
    vocab_data = json.load(f)

VOCAB_SIZE      = vocab_data['vocab_size']
MAX_LENGTH      = vocab_data['max_length']
WORD_TO_INDEX   = vocab_data['word_to_index']
INDEX_TO_WORD   = {int(k): v for k, v in vocab_data['index_to_word'].items()}
FEATURE_DIR     = "../../../dataset/features"

Helper: Keras Weight Extractor

In [ ]:
def build_keras_reader(num_layers, hidden_size, embed_dim=256):
    image_input = tf.keras.Input(shape=(2048,), name="image_input")
    image_emb   = tf.keras.layers.Dense(embed_dim, activation='relu', name="proj_dense")(image_input)
    image_emb   = tf.keras.layers.Reshape((1, embed_dim))(image_emb)

    caption_input = tf.keras.Input(shape=(MAX_LENGTH,), name="caption_input")
    caption_emb   = tf.keras.layers.Embedding(VOCAB_SIZE, embed_dim, mask_zero=True, name="embedding")(caption_input)

    merged = tf.keras.layers.Concatenate(axis=1)([image_emb, caption_emb])

    x = merged
    for i in range(num_layers):
        x = tf.keras.layers.SimpleRNN(hidden_size, return_sequences=True, name=f"rnn_{i}")(x)

    x = tf.keras.layers.Lambda(lambda t: t[:, 1:, :])(x)
    
    output = tf.keras.layers.TimeDistributed(
        tf.keras.layers.Dense(VOCAB_SIZE, activation='softmax'),
        name="time_dist_out"
    )(x)

    return tf.keras.Model(inputs=[image_input, caption_input], outputs=output)

def extract_weights_to_dict(keras_model, num_layers):
    params = {}
    
    params['proj_w'], params['proj_b'] = keras_model.get_layer("proj_dense").get_weights()
    params['embedding'] = keras_model.get_layer("embedding").get_weights()
    
    for i in range(num_layers):
        # [kernel (W_xh), recurrent_kernel (W_hh), bias (b_h)]
        params[f'rnn_{i}'] = keras_model.get_layer(f"rnn_{i}").get_weights()
        
    time_dist_layer = keras_model.get_layer("time_dist_out")
    params['out_w'], params['out_b'] = time_dist_layer.get_weights()
    
    return params

Evaluation Function [ Keras ]

In [ ]:
def generate_caption_keras(model, image_feature, max_length, word_to_index, index_to_word):
    in_text = '<start>'
    for i in range(max_length):
        sequence = [word_to_index.get(w, 0) for w in in_text.split()]
        sequence = tf.keras.preprocessing.sequence.pad_sequences([sequence], maxlen=max_length, padding='post')
        
        inputs = {
            "image_input": np.expand_dims(image_feature, axis=0),
            "caption_input": sequence
        }
        yhat = model.predict(inputs, verbose=0)
        
        current_len = len(in_text.split())
        yhat_idx = np.argmax(yhat[0, current_len - 1, :])
        word = index_to_word.get(yhat_idx, '')
        
        if word == '<end>' or not word:
            break
        in_text += ' ' + word
        
    return in_text.replace('<start> ', '').strip()

def evaluate_metrics(model, test_df, feature_dir, word_to_index, index_to_word, max_length):
    actual, predicted = [], []
    test_grouped = test_df.groupby('image')['caption'].apply(list).to_dict()
    
    for img_name, captions in list(test_grouped.items())[:2]: # mohon maap tapi kalau 1000 kelamaan 
    # for img_name, captions in list(test_grouped.items()): 
        feat_path = os.path.join(feature_dir, img_name + ".npy")
        feature = np.load(feat_path)
        
        yhat = generate_caption_keras(model, feature, max_length, word_to_index, index_to_word)
        references = [c.replace('<start> ', '').replace(' <end>', '').split() for c in captions]
        
        actual.append(references)
        predicted.append(yhat.split())
    
    smoothie = SmoothingFunction().method4
    bleu_4 = corpus_bleu(actual, predicted, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smoothie)
    
    meteor_scores = [meteor_score(ref, hyp) for ref, hyp in zip(actual, predicted)]
    return bleu_4, np.mean(meteor_scores)

Evaluation Function [ Scratch ]

In [ ]:
def generate_caption_scratch(model_scratch, image_feature, max_length, word_to_index, index_to_word):
    in_text = '<start>'
    for i in range(max_length):
        sequence = [word_to_index.get(w, 0) for w in in_text.split()]
        sequence = tf.keras.preprocessing.sequence.pad_sequences([sequence], maxlen=max_length, padding='post')
        
        # Prepare Input -> dimensi batch (batch_size=1)
        img_input = np.expand_dims(image_feature, axis=0)
        
        # Forward Pass
        probs = model_scratch.forward(img_input, sequence)
        
        current_len = len(in_text.split())
        yhat_idx = np.argmax(probs[0, current_len - 1, :])
        word = index_to_word.get(yhat_idx, '')
        
        if word == '<end>' or not word:
            break
        in_text += ' ' + word
        
    return in_text.replace('<start> ', '').strip()

def evaluate_scratch_metrics(model_scratch, test_df, feature_dir, word_to_index, index_to_word, max_length):
    actual, predicted = [], []
    test_grouped = test_df.groupby('image')['caption'].apply(list).to_dict()
    
    for img_name, captions in list(test_grouped.items())[:2]: # mohon maap tapi kalau 1000 kelamaan 
    # for img_name, captions in list(test_grouped.items()): 
        feat_path = os.path.join(feature_dir, img_name + ".npy")
        feature = np.load(feat_path)
        
        yhat = generate_caption_scratch(model_scratch, feature, max_length, word_to_index, index_to_word)
        references = [c.replace('<start> ', '').replace(' <end>', '').split() for c in captions]
        
        actual.append(references)
        predicted.append(yhat.split())
    
    smoothie = SmoothingFunction().method4
    bleu_4 = corpus_bleu(actual, predicted, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smoothie)
    meteor_scores = [meteor_score(ref, hyp) for ref, hyp in zip(actual, predicted)]
    
    return bleu_4, np.mean(meteor_scores)

Data Preparation and Splitting

In [ ]:
# Data Preparation
df_captions = pd.read_csv("../../../dataset/captions.txt")
df_captions = df_captions.dropna()
df_captions['caption'] = "<start> " + df_captions['caption'].str.lower() + " <end>"

# Splitting
unique_images = df_captions['image'].unique()
train_val_imgs, test_imgs = train_test_split(unique_images, test_size=1000, random_state=42)
test_df = df_captions[df_captions['image'].isin(test_imgs)]

print(f"Data Test disiapkan: {len(test_imgs)} gambar.")

Sanity Check

In [ ]:
test_grouped = test_df.groupby('image')['caption'].apply(list).to_dict()
version_name = f"rnn_l1_h256"
weight_dir = f"../weights/"
weight_path = os.path.join(weight_dir, f"{version_name}.weights.h5")

tf.keras.backend.clear_session()
keras_model = build_keras_reader(num_layers=1, hidden_size=256)
keras_model.load_weights(weight_path)

weights_dict = extract_weights_to_dict(keras_model, 1)
scratch_model = DecoderRNN(vocab_size=VOCAB_SIZE, embed_dim=256, hidden_size=256, num_layers=1)
scratch_model.set_model_params(weights_dict)

# Ambil 1 gambar acak dari test set
sample_img_name = list(test_grouped.keys())[0]
sample_feat_path = os.path.join(FEATURE_DIR, sample_img_name + ".npy")
sample_feature = np.load(sample_feat_path)

# Generate dari kedua model
keras_cap = generate_caption_keras(keras_model, sample_feature, MAX_LENGTH, WORD_TO_INDEX, INDEX_TO_WORD)
scratch_cap = generate_caption_scratch(scratch_model, sample_feature, MAX_LENGTH, WORD_TO_INDEX, INDEX_TO_WORD)

print(f"Gambar  : {sample_img_name}")
print(f"Keras   : {keras_cap}")
print(f"Scratch : {scratch_cap}")

Evaluation and Comparison

In [ ]:
# Hyperparameters
layers_variations = [1, 2, 3]
hidden_variations = [256, 512]

results_comparison = []

for n_layers in layers_variations:
    for h_size in hidden_variations:
        tf.keras.backend.clear_session()

        version_name = f"rnn_l{n_layers}_h{h_size}"
        weight_dir = f"../weights/"
        weight_path = os.path.join(weight_dir, f"{version_name}.weights.h5")
        
        print(f"\n{'='*60}")
        print(f"Evaluasi & Perbandingan Inferensi: {version_name}")
        print(f"{'='*60}")

        # ---------------------------------------------------------
        # EVALUASI KERAS
        # ---------------------------------------------------------
        keras_model = build_keras_reader(num_layers=n_layers, hidden_size=h_size)
        keras_model.load_weights(weight_path)
        
        print("-> Menjalankan inferensi Keras...")
        start_time_keras = time.time()
        keras_bleu, keras_meteor = evaluate_metrics(
            keras_model, test_df, FEATURE_DIR, WORD_TO_INDEX, INDEX_TO_WORD, MAX_LENGTH
        )
        keras_eval_time = time.time() - start_time_keras

        # ---------------------------------------------------------
        # EVALUASI FROM SCRATCH
        # ---------------------------------------------------------
        weights_dict = extract_weights_to_dict(keras_model, n_layers)
        
        scratch_model = DecoderRNN(vocab_size=VOCAB_SIZE, embed_dim=256, hidden_size=h_size, num_layers=n_layers)
        scratch_model.set_model_params(weights_dict)
        
        print("-> Menjalankan inferensi From Scratch...")
        start_time_scratch = time.time()
        scratch_bleu, scratch_meteor = evaluate_scratch_metrics(
            scratch_model, test_df, FEATURE_DIR, WORD_TO_INDEX, INDEX_TO_WORD, MAX_LENGTH
        )
        scratch_eval_time = time.time() - start_time_scratch
        
        # ---------------------------------------------------------
        # HASIL PERBANDINGAN
        # ---------------------------------------------------------
        print(f"\n--- Hasil Perbandingan ---")
        print(f"BLEU-4 : Keras = {keras_bleu:.4f} | Scratch = {scratch_bleu:.4f}")
        print(f"METEOR : Keras = {keras_meteor:.4f} | Scratch = {scratch_meteor:.4f}")
        print(f"Waktu  : Keras = {keras_eval_time:.2f}s | Scratch = {scratch_eval_time:.2f}s")
            
        results_comparison.append({
            "Variasi": version_name,
            "BLEU Keras": keras_bleu,
            "BLEU Scratch": scratch_bleu,
            "Waktu Keras (s)": round(keras_eval_time, 2),
            "Waktu Scratch (s)": round(scratch_eval_time, 2)
        })

In [ ]:
# Print rekap tabel
print("REKAP PERBANDINGAN INFERENSI (KERAS VS SCRATCH)")
print("\n" + "="*73)
df_results = pd.DataFrame(results_comparison)
print(df_results.to_string(index=False))

Visualization for Training and Validation

In [ ]:
# Definisi Variasi
layers_variations = [1, 2, 3]
hidden_variations = [256, 512]
weight_dir = "../weights/"

for n_layers in layers_variations:
    for h_size in hidden_variations:
        version_name = f"rnn_l{n_layers}_h{h_size}"
        meta_path = os.path.join(weight_dir, f"{version_name}_metadata.json")
        
        if not os.path.exists(meta_path):
            print(f"Melewati {version_name}: File metadata tidak ditemukan.")
            continue
            
        # Load data
        with open(meta_path, 'r') as f:
            metadata = json.load(f)
            
        history = metadata.get('history', {})

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        fig.suptitle(f"Training History - Arsitektur: {n_layers} Layer, {h_size} Hidden Size", fontsize=14, fontweight='bold')
        
        epochs = range(1, len(history['loss']) + 1)
        
        # Grafik Loss
        axes[0].plot(epochs, history['loss'], label='Training Loss', marker='o', color='blue')
        axes[0].plot(epochs, history['val_loss'], label='Validation Loss', marker='s', color='orange')
        axes[0].set_title('Loss Over Epochs')
        axes[0].set_xlabel('Epochs')
        axes[0].set_ylabel('Loss (Crossentropy)')
        axes[0].legend()
        axes[0].grid(True, linestyle='--', alpha=0.7)
        
        # Grafik Accuracy
        axes[1].plot(epochs, history['accuracy'], label='Training Accuracy', marker='o', color='green')
        axes[1].plot(epochs, history['val_accuracy'], label='Validation Accuracy', marker='s', color='red')
        axes[1].set_title('Accuracy Over Epochs')
        axes[1].set_xlabel('Epochs')
        axes[1].set_ylabel('Accuracy')
        axes[1].legend()
        axes[1].grid(True, linestyle='--', alpha=0.7)
            
        plt.tight_layout()
        plt.savefig(os.path.join(weight_dir, f"{version_name}_plot.png"), dpi=300, bbox_inches='tight')
        plt.show()